In [16]:
# ==========================================
# PROYECTO: EXTRACCIÓN DE DATOS URBANOS
# COMUNA: San Miguel, Santiago, Chile
# Autor: Enfoque Data Science + Ingeniería Urbana
# ==========================================

# 1. Instalación (ejecutar una vez)
# !pip install osmnx geopandas shapely fiona pyproj

In [17]:
# 2. Librerías
import osmnx as ox
import geopandas as gpd
import os

In [18]:
# 3. Configuración
ox.settings.log_console = True
ox.settings.use_cache = True

# Carpeta de salida
output_dir = "san_miguel_capas"
os.makedirs(output_dir, exist_ok=True)

# Área de estudio
place_name = "San Miguel, Santiago, Chile"

# Sistema de coordenadas proyectado (Chile)
CRS_PROY = "EPSG:32719"

print("Descargando datos para:", place_name)

Descargando datos para: San Miguel, Santiago, Chile


In [19]:
# ==========================================
# 4. RED VIAL (GRAFO)
# ==========================================
G = ox.graph_from_place(place_name, network_type="drive")
nodes, edges = ox.graph_to_gdfs(G)

edges = edges.to_crs(CRS_PROY)
nodes = nodes.to_crs(CRS_PROY)

edges.to_file(f"{output_dir}/red_vial.shp")
nodes.to_file(f"{output_dir}/nodos_viales.shp")

print("✔ Red vial descargada")

✔ Red vial descargada


C:\Users\anton\AppData\Local\Temp\ipykernel_7696\3843960651.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  nodes.to_file(f"{output_dir}/nodos_viales.shp")
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'street_count' to 'street_cou'
  ogr_write(


In [20]:
# ==========================================
# 5. NODOS CRÍTICOS (INTERSECCIONES)
# ==========================================
nodos_criticos = nodes[nodes["street_count"] >= 4]
nodos_criticos.to_file(f"{output_dir}/nodos_criticos.shp")

print("✔ Nodos críticos identificados")

✔ Nodos críticos identificados


C:\Users\anton\AppData\Local\Temp\ipykernel_7696\4256245820.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  nodos_criticos.to_file(f"{output_dir}/nodos_criticos.shp")
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'street_count' to 'street_cou'
  ogr_write(


In [21]:
# ==========================================
# 6. EQUIPAMIENTO URBANO (POIs)
# ==========================================
tags_pois = {
    "amenity": True,
    "shop": True,
    "tourism": True
}

pois = ox.features_from_place(place_name, tags_pois)

if not pois.empty:
    pois = pois.to_crs(CRS_PROY)
    # Convertir todas las geometrías a centroides (puntos)
    pois["geometry"] = pois.geometry.centroid
    pois.to_file(f"{output_dir}/equipamiento_urbano.shp")
    print("✔ POIs descargados")
else:
    print("⚠ No se encontraron POIs")

C:\Users\anton\AppData\Local\Temp\ipykernel_7696\315722030.py:16: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  pois.to_file(f"{output_dir}/equipamiento_urbano.shp")
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:city' to 'addr_city'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:full' to 'addr_full'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:housenumber' to 'addr_house'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:street' to 'addr_stree'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: R

✔ POIs descargados


In [22]:
# ==========================================
# 7. PARADEROS DE BUS
# ==========================================
tags_bus = {"highway": "bus_stop"}
try:
    bus_stops = ox.features_from_place(place_name, tags_bus)
    if not bus_stops.empty:
        bus_stops = bus_stops.to_crs(CRS_PROY)
        bus_stops["geometry"] = bus_stops.geometry.centroid
        bus_stops.to_file(f"{output_dir}/paraderos.shp")
        print("✔ Paraderos descargados")
except Exception as e:
    print(f"⚠ No se encontraron Paraderos ({type(e).__name__})")

C:\Users\anton\AppData\Local\Temp\ipykernel_7696\3114773331.py:10: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  bus_stops.to_file(f"{output_dir}/paraderos.shp")
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'attribution' to 'attributio'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'is_in:city' to 'is_in_city'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'network:wikidata' to 'network_wi'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'public_transport' to 'public_tra'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:

✔ Paraderos descargados


In [23]:
# ==========================================
# 8. INFRAESTRUCTURA PEATONAL
# ==========================================
tags_walk = {"highway": ["footway", "pedestrian", "path"]}
try:
    walkways = ox.features_from_place(place_name, tags_walk)
    if not walkways.empty:
        walkways = walkways.to_crs(CRS_PROY)
        walkways.to_file(f"{output_dir}/infraestructura_peatonal.shp")
        print("✔ Infraestructura peatonal descargada")
except Exception as e:
    print(f"⚠ No se encontró infraestructura peatonal ({type(e).__name__})")

✔ Infraestructura peatonal descargada


C:\Users\anton\AppData\Local\Temp\ipykernel_7696\195334657.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  walkways.to_file(f"{output_dir}/infraestructura_peatonal.shp")
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'crossing:markings' to 'crossing_m'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'motor_vehicle' to 'motor_vehi'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'crossing:island' to 'crossing_i'
  ogr_write(


In [24]:
# ==========================================
# 9. CICLOVÍAS
# ==========================================
tags_cycle = {"cycleway": True}
try:
    cycleways = ox.features_from_place(place_name, tags_cycle)
    if not cycleways.empty:
        cycleways = cycleways.to_crs(CRS_PROY)
        cycleways.to_file(f"{output_dir}/ciclovias.shp")
        print("✔ Ciclovías descargadas")
except Exception as e:
    print(f"⚠ No se encontraron Ciclovías ({type(e).__name__})")

⚠ No se encontraron Ciclovías (InsufficientResponseError)


In [25]:
# ==========================================
# 10. USO DE SUELO
# ==========================================
tags_landuse = {"landuse": True}
try:
    landuse = ox.features_from_place(place_name, tags_landuse)
    if not landuse.empty:
        landuse = landuse.to_crs(CRS_PROY)
        landuse.to_file(f"{output_dir}/uso_suelo.shp")
        print("✔ Uso de suelo descargado")
except Exception as e:
    print(f"⚠ No se encontró Uso de suelo ({type(e).__name__})")

✔ Uso de suelo descargado


C:\Users\anton\AppData\Local\Temp\ipykernel_7696\3321248959.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  landuse.to_file(f"{output_dir}/uso_suelo.shp")
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:city' to 'addr_city'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:housenumber' to 'addr_house'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'addr:street' to 'addr_stree'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'construction' to 'constructi'
  ogr_write(
C:\Users\anton\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: Runt

In [26]:
# ==========================================
# 11. LIMPIEZA BÁSICA (OPCIONAL)
# ==========================================
# Eliminar geometrías inválidas
def limpiar_gdf(gdf):
    return gdf[gdf.geometry.notnull()]

In [27]:
# ==========================================
# 12. RESUMEN FINAL
# ==========================================
print("\\n===== RESUMEN =====")
print(f"Nodos totales: {len(nodes)}")
print(f"Calles totales: {len(edges)}")
print(f"Nodos críticos: {len(nodos_criticos)}")

print("\\nArchivos guardados en carpeta:", output_dir)
print("Proceso completado correctamente")

\n===== RESUMEN =====
Nodos totales: 1171
Calles totales: 2312
Nodos críticos: 438
\nArchivos guardados en carpeta: san_miguel_capas
Proceso completado correctamente
